# 09 — Improved NCCS Historical Risk Modeling (2010–2024)

**Goal:** Rank nonprofit filings using current-return financial/governance features and historical context, while improving labeled-data coverage over Notebook 08's complete-case baseline.

**Target:** `fraud_label` is **self-reported significant diversion of assets**, **not verified fraud**. A risk score is a review priority, not an allegation. Unknown labels stay unknown; they can be scored but not used as ground truth.

**Inputs:** `data/processed/nccs_longitudinal_features_v1.parquet` produced by Notebook 08. Notebook 08A supplies structural audits and aggregate duplicate-label findings. No raw NCCS re-download or Notebook 08 rerun is required if this file is current.

**Temporal policy:** train 2010–2020; select model using validation 2021–2022; evaluate **once** on test 2023–2024. The same labeled test rows are used for all improved-model comparisons. Historical features must not use later tax years. For an actual filing-time system, validate filing availability by submission date separately; the current data is retrospective.

## 1. Setup and input checks
Run from the repository root **or** the `notebooks/` directory. Dependencies: `duckdb`, `pandas`, `numpy`, `scikit-learn`, `pyarrow`, `matplotlib`, `joblib`. On your Mac, select the same `irs990` notebook kernel used for Notebook 08.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
import sklearn
from IPython.display import display

SEED = 42
from pathlib import Path

# Locate the project root, whether running from notebooks/ or the root folder
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT = ROOT / "data" / "processed" / "nccs_longitudinal_features_v1.parquet"
OUT = ROOT / "data" / "processed" / "notebook09"

OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Input file:", INPUT)
print("File exists:", INPUT.is_file())
OUT.mkdir(parents=True, exist_ok=True)
assert INPUT.is_file(), f"Missing {INPUT}. Run Notebook 08 feature exports first."

con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET memory_limit = '6GB'")
con.execute(f"CREATE OR REPLACE VIEW features AS SELECT * FROM read_parquet('{str(INPUT).replace(chr(39), chr(39) + chr(39))}')")
print("Input:", INPUT)
print("Output directory:", OUT)
print("scikit-learn:", sklearn.__version__)
print("Columns:", con.execute("DESCRIBE features").df().column_name.tolist())

## 2. Define predictors and enforce a target-leakage boundary
Only numeric features built from the current or preceding filings are considered. **Never** include `fraud_label`, `raw_disclosure`, `OBJECTID`, EIN, or later-year disclosures as predictors. This is a *retrospective* current-return score; an operational point-in-time audit is still needed. The existing `rev_growth_3yr` name denotes current-versus-two-years-prior change, not a CAGR.

In [ ]:
FEATURES = [
    "total_assets", "total_liabilities", "total_revenue", "total_expenses",
    "net_income", "debt_to_asset_ratio", "voting_members", "independent_members",
    "pct_independent_board", "rev_growth_3yr", "asset_growth_3yr",
    "rev_volatility_3yr", "avg_net_income_3yr", "flag_small_board",
    "flag_minority_independent_board", "flag_board_independence_drop",
    "flag_sustained_deficit", "has_three_consecutive_years",
]
IDENTIFIERS = ["OBJECTID", "ein", "tax_year", "fraud_label"]
available = set(con.execute("DESCRIBE features").df().column_name)
assert set(FEATURES + IDENTIFIERS) <= available, sorted(set(FEATURES + IDENTIFIERS) - available)
assert "fraud_label" not in FEATURES

# Report the exact populations BEFORE model filtering.
coverage = con.execute("""
    SELECT CASE WHEN tax_year BETWEEN 2010 AND 2020 THEN 'train'
                WHEN tax_year BETWEEN 2021 AND 2022 THEN 'validation'
                WHEN tax_year BETWEEN 2023 AND 2024 THEN 'test' ELSE 'other' END AS split,
           COUNT(*) AS all_filings,
           COUNT(*) FILTER (WHERE fraud_label IN (0,1)) AS known_labels,
           COUNT(*) FILTER (WHERE fraud_label = 1) AS positive_labels,
           COUNT(*) FILTER (WHERE fraud_label IS NULL) AS unknown_labels
    FROM features GROUP BY 1 ORDER BY 1
""").df()
display(coverage)
coverage.to_csv(OUT / "01_label_coverage.csv", index=False)

## 3. Measure complete-case losses without discarding incomplete predictors
This is the main coverage improvement. Missing **targets** are excluded from supervised fitting and evaluation; missing **predictors** are retained and handled by train-fitted preprocessing. The coverage table distinguishes these causes.

In [ ]:
# A row is complete only if ALL selected predictors are finite and non-NULL.
finite_checks = " AND ".join(
    f'("{c}" IS NOT NULL AND isfinite(TRY_CAST("{c}" AS DOUBLE)))' for c in FEATURES
)
coverage_detail = con.execute(f"""
    SELECT CASE WHEN tax_year BETWEEN 2010 AND 2020 THEN 'train'
                WHEN tax_year BETWEEN 2021 AND 2022 THEN 'validation'
                WHEN tax_year BETWEEN 2023 AND 2024 THEN 'test' ELSE 'other' END AS split,
           COUNT(*) AS total,
           COUNT(*) FILTER (WHERE fraud_label IN (0,1)) AS labeled,
           COUNT(*) FILTER (WHERE fraud_label IN (0,1) AND ({finite_checks})) AS labeled_complete_case,
           COUNT(*) FILTER (WHERE fraud_label IN (0,1) AND NOT ({finite_checks})) AS labeled_with_missing_features,
           COUNT(*) FILTER (WHERE fraud_label IS NULL) AS unknown_target
    FROM features GROUP BY 1 ORDER BY 1
""").df()
display(coverage_detail)
coverage_detail.to_csv(OUT / "02_model_coverage.csv", index=False)

## 4. Load labeled train, validation and test rows
Only required columns are loaded. The entire **labeled** validation and test populations are kept; train may be sampled **only for fitting** to control memory and runtime. All positives in training are retained. If memory is limited, lower `MAX_TRAIN_ROWS` without changing validation or test.

In [ ]:
SELECT_COLS = IDENTIFIERS + FEATURES
quoted = ", ".join(f'"{c}"' for c in SELECT_COLS)

def load_years(lo, hi, known_only=True):
    where = "AND fraud_label IN (0, 1)" if known_only else ""
    frame = con.execute(f"""
        SELECT {quoted} FROM features
        WHERE tax_year BETWEEN ? AND ? {where}
        ORDER BY tax_year, ein, OBJECTID
    """, [lo, hi]).df()
    for c in FEATURES:
        frame[c] = pd.to_numeric(frame[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float32")
    return frame

train_all = load_years(2010, 2020)
val = load_years(2021, 2022)
test = load_years(2023, 2024)
for name, frame in [("train", train_all), ("validation", val), ("test", test)]:
    assert frame["fraud_label"].isin([0, 1]).all()
    assert not frame.duplicated(["ein", "tax_year"]).any(), f"Duplicate EIN/year in {name}"
    print(name, f"{len(frame):,} labeled rows", f"{int(frame.fraud_label.sum()):,} positives")

## 5. Fit-sample policy and preprocessing
Training uses all positive examples plus a reproducible random sample of negatives if the labeled train population exceeds `MAX_TRAIN_ROWS`. This sampling changes the class prevalence, so predicted probabilities **must not be interpreted as calibrated diversion probabilities**. Models are compared as rankings. Validation and test remain unsampled. Imputation, missingness indicators, and scaling are fitted on training data only.

In [ ]:
MAX_TRAIN_ROWS = 500_000
MAX_HGB_ROWS = 250_000

positives = train_all.loc[train_all.fraud_label.eq(1)]
negatives = train_all.loc[train_all.fraud_label.eq(0)]
if len(positives) > MAX_TRAIN_ROWS:
    raise ValueError("MAX_TRAIN_ROWS smaller than positive training population")
n_neg = min(len(negatives), MAX_TRAIN_ROWS - len(positives))
train_fit = pd.concat([
    positives,
    negatives.sample(n=n_neg, random_state=SEED) if n_neg < len(negatives) else negatives,
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Training fit sample: {len(train_fit):,}; positives: {int(train_fit.fraud_label.sum()):,}")
print("Full training labeled population:", f"{len(train_all):,}")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# SimpleImputer(add_indicator=True) makes missingness observable to the model.
# StandardScaler is important for SGD logistic regression with dollar-valued features.
logistic = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("scale", StandardScaler()),
    ("model", SGDClassifier(loss="log_loss", alpha=0.0001, class_weight="balanced",
                            max_iter=1500, tol=1e-3, random_state=SEED)),
])

# Tree-based benchmark; uses the same imputation/missingness features.
hgb = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
    ("model", HistGradientBoostingClassifier(max_iter=160, max_leaf_nodes=15,
                                              learning_rate=0.06, l2_regularization=2.0,
                                              random_state=SEED)),
])

X_val = val[FEATURES]
y_val = val.fraud_label.astype("int8").to_numpy()
X_test = test[FEATURES]
y_test = test.fraud_label.astype("int8").to_numpy()

## 6. Train improved candidates; select on validation only
Selection metric: **average precision (AP)**, which is more interpretable for rare positives than accuracy. Also report trapezoidal PR-AUC, ROC-AUC, and top-K capture. Both candidates use the same validation records. No test labels are used to select the model.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, auc, roc_auc_score

def ranking_metrics(y, scores, ks=(100, 250, 500, 1000)):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    assert len(y) == len(scores) and np.isfinite(scores).all()
    p, r, _ = precision_recall_curve(y, scores)
    result = {
        "n": len(y), "positives": int(y.sum()),
        "prevalence": float(y.mean()),
        "average_precision": float(average_precision_score(y, scores)),
        "trapezoid_pr_auc": float(auc(r, p)),
        "roc_auc": float(roc_auc_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
    }
    ranking = np.argsort(-scores, kind="stable")
    for k in ks:
        actual_k = min(k, len(y))
        found = int(y[ranking[:actual_k]].sum())
        result[f"found_at_{k}"] = found
        result[f"precision_at_{k}"] = found / actual_k if actual_k else np.nan
    return result

candidates = {}
print("Fitting SGD logistic regression...")
logistic.fit(train_fit[FEATURES], train_fit.fraud_label.astype(int))
candidates["SGD Logistic + imputation"] = logistic

# Use a smaller sample for the slower tree model, retaining all available positives.
if len(train_fit) > MAX_HGB_ROWS:
    hgb_pos = train_fit.loc[train_fit.fraud_label.eq(1)]
    hgb_neg = train_fit.loc[train_fit.fraud_label.eq(0)]
    n = MAX_HGB_ROWS - len(hgb_pos)
    hgb_fit = pd.concat([hgb_pos, hgb_neg.sample(n=n, random_state=SEED)], ignore_index=True)
else:
    hgb_fit = train_fit
print("Fitting histogram gradient boosting...")
hgb.fit(hgb_fit[FEATURES], hgb_fit.fraud_label.astype(int))
candidates["HistGradientBoosting + imputation"] = hgb

val_scores = {}
validation_rows = []
for name, model in candidates.items():
    scores = model.predict_proba(X_val)[:, 1]
    val_scores[name] = scores
    validation_rows.append({"model": name, **ranking_metrics(y_val, scores)})
validation_results = pd.DataFrame(validation_rows).sort_values("average_precision", ascending=False)
display(validation_results)
validation_results.to_csv(OUT / "03_validation_model_comparison.csv", index=False)
SELECTED_NAME = validation_results.iloc[0]["model"]
SELECTED_MODEL = candidates[SELECTED_NAME]
print("Selected on validation AP:", SELECTED_NAME)

## 7. Evaluate selected model on the untouched 2023–2024 test set
The holdout is evaluated only after selection. For transparency, evaluate both improved candidates on the **same** test rows, but do not change the selected model based on test results.

In [ ]:
test_scores = {}
test_rows = []
for name, model in candidates.items():
    scores = model.predict_proba(X_test)[:, 1]
    test_scores[name] = scores
    test_rows.append({"model": name, "selected_on_validation": name == SELECTED_NAME,
                      **ranking_metrics(y_test, scores)})
test_results = pd.DataFrame(test_rows).sort_values("selected_on_validation", ascending=False)
display(test_results)
test_results.to_csv(OUT / "04_holdout_model_comparison.csv", index=False)

ranked_test = test[IDENTIFIERS].copy()
ranked_test["risk_score"] = test_scores[SELECTED_NAME]
ranked_test["risk_rank"] = ranked_test.risk_score.rank(ascending=False, method="first").astype(int)
ranked_test = ranked_test.sort_values("risk_rank")
ranked_test.to_parquet(OUT / "05_labeled_holdout_ranked.parquet", index=False)
print("Saved holdout ranking:", OUT / "05_labeled_holdout_ranked.parquet")

## 8. Compare with Notebook 08's complete-case baseline on the **same test records**
Notebook 08 used complete-case filtering. For a fair apples-to-apples comparison, train a new complete-case Logistic Regression baseline on 2010–2020, then evaluate **both** baseline and improved selected model on the **identical complete-case test subset**. Also retain the improved model's all-labeled-test results separately; these populations must not be conflated. The baseline is a methodological comparison, not an exact reproduction of Notebook 08's prior coefficients.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler

BASELINE_FEATURES = [
    "total_assets", "total_liabilities", "total_revenue", "total_expenses",
    "net_income", "debt_to_asset_ratio", "voting_members", "independent_members",
    "pct_independent_board", "rev_growth_3yr", "asset_growth_3yr",
    "rev_volatility_3yr", "avg_net_income_3yr", "flag_small_board",
    "flag_minority_independent_board", "flag_board_independence_drop",
]
base_train = train_fit.dropna(subset=BASELINE_FEATURES)
base_test_mask = test[BASELINE_FEATURES].notna().all(axis=1).to_numpy()
base_test = test.loc[base_test_mask]
print("Complete-case train:", len(base_train), "test:", len(base_test))
assert base_train.fraud_label.nunique() == 2 and base_test.fraud_label.nunique() == 2

baseline = Pipeline([
    ("scale", RobustScaler()),
    ("model", LogisticRegression(max_iter=600, class_weight="balanced", random_state=SEED)),
])
baseline.fit(base_train[BASELINE_FEATURES], base_train.fraud_label.astype(int))
baseline_scores = baseline.predict_proba(base_test[BASELINE_FEATURES])[:, 1]
improved_same_rows = test_scores[SELECTED_NAME][base_test_mask]
comparison = pd.DataFrame([
    {"model": "Complete-case LR baseline (retrained)", **ranking_metrics(base_test.fraud_label, baseline_scores)},
    {"model": f"Improved selected: {SELECTED_NAME}", **ranking_metrics(base_test.fraud_label, improved_same_rows)},
])
display(comparison)
comparison.to_csv(OUT / "06_same_population_baseline_comparison.csv", index=False)

## 9. Diagnose performance and coverage by tax year
These are **descriptive** holdout checks, not grounds for changing model selection. Compare the number of labeled filings with the larger pool of unknown labels. Small positive counts can make yearly metrics unstable.

In [ ]:
year_rows = []
for year, part in test.groupby("tax_year", sort=True):
    idx = test.index.get_indexer(part.index)
    score = test_scores[SELECTED_NAME][idx]
    year_rows.append({"tax_year": int(year), **ranking_metrics(part.fraud_label, score)})
by_year = pd.DataFrame(year_rows)
display(by_year)
by_year.to_csv(OUT / "07_holdout_by_year.csv", index=False)

missing_report = pd.DataFrame({
    "feature": FEATURES,
    "train_missing_pct": train_all[FEATURES].isna().mean().mul(100).values,
    "test_missing_pct": test[FEATURES].isna().mean().mul(100).values,
}).sort_values("test_missing_pct", ascending=False)
display(missing_report)
missing_report.to_csv(OUT / "08_feature_missingness.csv", index=False)

## 10. Score all 2024 filings, including unknown-label organizations
This is a separate **scoring** exercise, not supervised evaluation. A missing label is never converted to zero. Only one row per EIN/tax year is expected. These scores are **not calibrated probabilities**; use them for exploratory review only. To reduce memory, 2024 is loaded separately.

In [ ]:
all_2024 = load_years(2024, 2024, known_only=False)
assert not all_2024.duplicated(["ein", "tax_year"]).any()
all_2024_scores = SELECTED_MODEL.predict_proba(all_2024[FEATURES])[:, 1]
queue_2024 = all_2024[IDENTIFIERS].copy()
queue_2024["risk_score"] = all_2024_scores
queue_2024["risk_rank"] = queue_2024.risk_score.rank(ascending=False, method="first").astype(int)
queue_2024["label_status"] = np.where(queue_2024.fraud_label.isna(), "unknown", "known")
queue_2024 = queue_2024.sort_values("risk_rank")
queue_2024.to_parquet(OUT / "09_all_2024_investigation_queue.parquet", index=False)
print("2024 filings scored:", len(queue_2024))
print("2024 unknown labels scored:", int(queue_2024.fraud_label.isna().sum()))
print("Saved:", OUT / "09_all_2024_investigation_queue.parquet")
display(queue_2024.head(20))

## 11. Save reproducibility metadata and trained model
Model files use `joblib` (pickle-based); load only files you created/trust. Source and output file paths remain local. The notebook does **not** overwrite Notebook 08's baseline exports.

In [ ]:
import joblib
model_path = OUT / "selected_model.joblib"
joblib.dump(SELECTED_MODEL, model_path)
manifest = {
    "source_parquet": str(INPUT),
    "features": FEATURES,
    "target_definition": "Self-reported significant diversion of assets (not verified fraud)",
    "train_years": [2010, 2020], "validation_years": [2021, 2022], "test_years": [2023, 2024],
    "train_sample_cap": MAX_TRAIN_ROWS, "hgb_sample_cap": MAX_HGB_ROWS,
    "training_rows_used": len(train_fit),
    "selected_model": SELECTED_NAME,
    "selection_metric": "validation_average_precision",
    "seed": SEED,
    "sklearn_version": sklearn.__version__,
    "unknown_labels_excluded_from_fit_and_evaluation": True,
    "missing_predictors_imputed_from_train_only": True,
    "scores_are_calibrated_probabilities": False,
    "known_limitations": [
        "Retrospective tax-year split is not a submission-date point-in-time simulation",
        "67 conflicting known-label duplicate EIN/year groups in Notebook 08A",
        "512 mixed known/unknown groups selected with unknown label in Notebook 08A",
        "Source disclosures are not verified fraud outcomes",
        "Training negative downsampling may distort probability calibration",
    ],
}
(OUT / "model_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Model:", model_path)
print("Manifest:", OUT / "model_manifest.json")

In [ ]:
# Audit raw NCCS values against final model features

from pathlib import Path
import duckdb
import pandas as pd

audit_con = duckdb.connect()
audit_con.execute("SET threads = 4")

source_dir = ROOT / "data/external/nccs/efile_v3_1"
feature_file = ROOT / "data/processed/nccs_longitudinal_features_v1.parquet"

all_files = list(source_dir.rglob("*.parquet"))

summary_files = [
    str(p) for p in all_files
    if "SUMMARY" in p.name.upper()
]

governance_files = [
    str(p) for p in all_files
    if "GOVERNANCE" in p.name.upper()
]

print("Total Parquet files:", len(all_files))
print("SUMMARY files:", len(summary_files))
print("GOVERNANCE files:", len(governance_files))

print("Example SUMMARY:", summary_files[:2])
print("Example GOVERNANCE:", governance_files[:2])

assert summary_files, "No SUMMARY Parquet files found"
assert governance_files, "No GOVERNANCE Parquet files found"

audit_con.execute(f"""
    CREATE VIEW raw_summary AS
    SELECT
        OBJECTID,
        TRY_CAST(F9_01_NAFB_ASSET_TOT_EOY AS DOUBLE) AS total_assets,
        TRY_CAST(F9_01_NAFB_LIAB_TOT_EOY AS DOUBLE) AS total_liabilities,
        TRY_CAST(F9_01_REV_TOT_CY AS DOUBLE) AS total_revenue,
        TRY_CAST(F9_01_EXP_TOT_CY AS DOUBLE) AS total_expenses
    FROM read_parquet({repr(summary_files)}, union_by_name=true)
""")

audit_con.execute(f"""
    CREATE VIEW raw_governance AS
    SELECT
        OBJECTID,
        TRY_CAST(F9_06_GVRN_NUM_VOTING_MEMB AS DOUBLE) AS voting_members,
        TRY_CAST(F9_06_GVRN_NUM_VOTING_MEMB_IND AS DOUBLE) AS independent_members
    FROM read_parquet({repr(governance_files)}, union_by_name=true)
""")

audit_con.execute(f"""
    CREATE VIEW final_features AS
    SELECT *
    FROM read_parquet('{feature_file.as_posix()}')
""")

fields = [
    ("total_assets", "raw_summary"),
    ("total_liabilities", "raw_summary"),
    ("total_revenue", "raw_summary"),
    ("total_expenses", "raw_summary"),
    ("voting_members", "raw_governance"),
    ("independent_members", "raw_governance"),
]

results = []

for field, source in fields:
    result = audit_con.execute(f"""
        SELECT
            COUNT(*) AS total_filings,
            COUNT(*) FILTER (
                WHERE r.OBJECTID IS NULL
            ) AS missing_source_record,
            COUNT(*) FILTER (
                WHERE r.OBJECTID IS NOT NULL
                  AND r.{field} IS NULL
            ) AS source_null,
            COUNT(*) FILTER (
                WHERE r.{field} IS NULL
                  AND f.{field} = 0
            ) AS source_null_final_zero,
            COUNT(*) FILTER (
                WHERE r.{field} IS NOT NULL
                  AND f.{field} IS NULL
            ) AS source_known_final_null,
            COUNT(*) FILTER (
                WHERE r.{field} IS NOT NULL
                  AND f.{field} IS NOT NULL
                  AND r.{field} <> f.{field}
            ) AS value_mismatch
        FROM final_features f
        LEFT JOIN {source} r
            ON f.OBJECTID = r.OBJECTID
    """).df().iloc[0].to_dict()

    result["feature"] = field
    results.append(result)

audit_results = pd.DataFrame(results)

display(audit_results[
    [
        "feature",
        "total_filings",
        "missing_source_record",
        "source_null",
        "source_null_final_zero",
        "source_known_final_null",
        "value_mismatch",
    ]
])

audit_results.to_csv(
    OUT / "source_completeness_audit.csv",
    index=False
)

## Interpretation and next steps

1. **Coverage:** Notebook 09 expanded the labeled 2023–2024 test population to 708,494 filings by imputing missing predictors rather than excluding incomplete records. An additional 205,447 filings with unknown disclosure labels were scored in the 2024 investigation queue but remain excluded from supervised evaluation metrics.

2. **Model performance:** Histogram Gradient Boosting was selected using 2021–2022 validation Average Precision. It achieved a holdout ROC-AUC of approximately 0.658 and Average Precision of 0.00246. However, the complete-case Logistic Regression baseline identified more positive disclosures among the highest-ranked filings on the matched test population. Notebook 09B will investigate alternative models, training sample sizes, feature subsets, and ranking-based evaluation.

3. **Fairness of comparison:** Use `06_same_population_baseline_comparison.csv` for matched-population comparisons. Do not directly compare the all-labeled test results against Notebook 08's complete-case results.

4. **Source-data integrity:** An initial raw-to-feature audit compared six financial and governance fields across 5,964,599 deduplicated filings. No missing-source-to-zero conversions or non-null value mismatches were detected. However, substantial source missingness remains, particularly for assets, liabilities, and governance fields. Notebook 09A provides additional checks of feature integrity, historical missingness, governance flags, and model preprocessing.

5. **Label uncertainty:** Notebook 08A identified 67 duplicate EIN/year groups with conflicting known disclosures and 512 mixed groups whose selected label is unknown. Future work should export the affected EIN/year keys and evaluate model sensitivity to excluding these records.

6. **Point-in-time audit:** The current 2024 investigation queue is retrospective. Before using it for filing-time decisions, verify that all features and selected filings were available as of the intended scoring date. Amended filings and later submissions may introduce retrospective selection bias.

7. **Risk-score interpretation:** Model scores are intended for relative ranking, not as calibrated probabilities of fraud. The target represents self-reported significant diversion of assets, not independently verified misconduct. A high risk score should never be interpreted as evidence of wrongdoing.

8. **Next steps:**
   - **Notebook 09A:** Complete source-data and feature-integrity verification.
   - **Notebook 09B:** Optimize model selection and top-K ranking performance using validation data.
   - **Notebook 10:** Develop peer benchmarking, feature attribution, and analyst-facing risk explanations.

**Important:** Notebook 09 establishes a reproducible modeling baseline and a ranked investigation queue covering 561,420 filings for tax year 2024. The model's current ranking performance remains limited, and further validation is required before using its scores to prioritize real-world investigations.